In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import sys

# Habilitar import do módulo config.py em src/
sys.path.insert(0, str((Path('../../..').resolve() / 'src')))

from utils.monthly_data import load_price_series
from config import (
    DATA_DIR,
    OUTPUT_DIR,
    REGIOES,
    PRODUTOS as PRODUTOS_CONFIG,
    CHART_DPI,
    DEFAULT_FORECAST_HORIZON,
    CHART_GRID_ALPHA,
 )


# Gráficos de previsão do preço dos produtos da cesta básica

## Configurações

In [ ]:
# Configurações base (sem .conf)
cidades = list(REGIOES)
produtos = list(PRODUTOS_CONFIG)
forecast_horizon = DEFAULT_FORECAST_HORIZON

# Mantido para preservar a escala visual histórica dos gráficos
quantidades = {'acucar':3.0, 'arroz':3.6, 'banana':7.5,  'cafe':0.3, 'carne':4.5, 'farinha':3.0, 'feijao':4.5, 'leite':6.0,
               'manteiga':0.75, 'oleo':1.0, 'pao':6.0, 'tomate':12.0  }

# Caminhos de previsões
ios_path2 = OUTPUT_DIR / 'previsoes_produtos' / 'ilheus'
itb_path2 = OUTPUT_DIR / 'previsoes_produtos' / 'itabuna'

# Detectar modelo a partir dos arquivos de previsão existentes
modelos_disponiveis = ['RNN', 'LSTM', 'CNN']
modelo = None
for m in modelos_disponiveis:
    amostra = ios_path2 / f'previsao_{m}_acucar_ilheus.json'
    if amostra.exists():
        modelo = m
        break
if modelo is None:
    modelo = 'RNN'

# Construir labels de meses e ano com base no histórico da cesta básica
MESES_PT = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

def _mes_para_numero(valor):
    if pd.isna(valor):
        return None
    if isinstance(valor, (int, float)):
        mes = int(valor)
        return mes if 1 <= mes <= 12 else None
    texto = str(valor).strip().lower()
    mapa = {
        'jan': 1, 'janeiro': 1,
        'fev': 2, 'fevereiro': 2,
        'mar': 3, 'marco': 3, 'março': 3,
        'abr': 4, 'abril': 4,
        'mai': 5, 'maio': 5,
        'jun': 6, 'junho': 6,
        'jul': 7, 'julho': 7,
        'ago': 8, 'agosto': 8,
        'set': 9, 'setembro': 9,
        'out': 10, 'outubro': 10,
        'nov': 11, 'novembro': 11,
        'dez': 12, 'dezembro': 12
    }
    return mapa.get(texto[:3], mapa.get(texto))

df_hist = load_price_series('ilheus', 'cesta_basica')
datas_hist = pd.to_datetime(df_hist['data']).tail(9).tolist()

meses_anteriores = [MESES_PT[d.month - 1] for d in datas_hist]
datas_prev = [datas_hist[-1] + pd.DateOffset(months=i) for i in range(1, forecast_horizon + 1)]
meses_previstos = [MESES_PT[d.month - 1] for d in datas_prev]
ano_previsao = datas_prev[-1].year

line_size = 3
xlabel = meses_anteriores + meses_previstos

marks = ['s', '.', 'v', 'p', 'p', 'X', '*', 'D', '^', '8', 'P', '>']
colors_prev = ['#0004c7', '#c9261a', '#c1c718', '#57210a', '#6fad11', '#039eff', '#ffb100', '#00fa08', '#ff0044', '#210109', '#780c6d',
               '#b200b5']
colors_real = ['#7578ff', '#e6837c', '#c0c28c', '#634b41', '#aac77f', '#aadbfa', '#f7d381', '#a2fca4', '#f79cb4', '#736e6f', '#854e7f',
               '#c877c9']
margens = [-1, -1, -2, 2, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5, 7, 6]
markers_size = [8, 17, 10, 10, 10, 10, 15, 8, 10, 10, 10, 10]

limite_y = 91


## Carregar valores reais e previstos 

In [ ]:
# capturando resultados de ilhéus
previsoes_ios = {}
previsoes_itb = {}

for produto in produtos:
    produto_path = ios_path2 / f"previsao_{modelo}_{produto}_{cidades[0]}.json"
    with open(produto_path,'r', encoding='utf-8') as file:
        previsao = json.load(file)
        valores = previsao[produto]
        if isinstance(valores, str):
            valores = valores.replace('[','').replace(']','').split(',')
            valores = list(map(float, valores))
        else:
            valores = [float(v) for v in valores]
        valores = [previsto/quantidades[produto] * 1000 for previsto in valores]
        previsoes_ios[produto] = valores

# capturando resultados de itabuna
for produto in produtos:
    produto_path = itb_path2 / f"previsao_{modelo}_{produto}_{cidades[1]}.json"
    with open(produto_path,'r', encoding='utf-8') as file:
        previsao = json.load(file)
        valores = previsao[produto]
        if isinstance(valores, str):
            valores = valores.replace('[','').replace(']','').split(',')
            valores = list(map(float, valores))
        else:
            valores = [float(v) for v in valores]
        valores = [previsto/quantidades[produto] * 1000 for previsto in valores]
        previsoes_itb[produto] = valores

# Capturando os valores reais
valores_reais_ios = {}
valores_reais_itb = {}

# Capturando os valores de Ilhéus
for produto in produtos:
    produto_path = reais_ios_path / f"{produto}_ilheus.xlsx"
    df = pd.read_excel(produto_path)
    temp = df['preco'].tail(9).to_list()
    temp = [valor/quantidades[produto] for valor in temp]
    valores_reais_ios[produto] = temp

# Capturando os valores de Itabuna
for produto in produtos:
    produto_path = reais_itb_path / f"{produto}_itabuna.xlsx"
    df = pd.read_excel(produto_path)
    temp = df['preco'].tail(9).to_list()
    temp = [valor/quantidades[produto] for valor in temp]
    valores_reais_itb[produto] = temp


## Plotar Gráfico Ilhéus

In [ ]:
plt.figure(figsize=(18, 10))

#Plot - Ilhéus
for produto, mark, color_prev, color_real, margem, marker_size in zip(produtos, marks, colors_prev, colors_real, margens, markers_size):
    plt.plot([8,9],[valores_reais_ios[produto][-1],previsoes_ios[produto][0]],":",color=color_prev,lw=line_size, markersize=marker_size)
    plt.plot([x for x in range(9)],valores_reais_ios[produto], marker=mark, color=color_real,lw=line_size, markersize=marker_size)
    plt.plot([9,10,11],previsoes_ios[produto],":" ,marker=mark, label=produto.title().replace('c','ç'),color=color_prev,lw=line_size,
             markersize=marker_size)

#----------------------------------------------------
plt.xticks([x for x in range(0,12)],xlabel,size=15)
plt.ylabel("Custo (R$)",size=20)
plt.yticks([y for y in range(0,limite_y,10)],[format(y,".2f").replace(".",",") for y in range(0,limite_y,10)],size=15)
plt.grid(which='major', axis='y', alpha=CHART_GRID_ALPHA)
#----------
plt.title("Previsão do valor dos produtos em Ilhéus", size=15)
plt.legend(ncol=4)
chart_path = OUTPUT_DIR / 'figure' / 'produtos_ilheus' / f'previsao_produtos_{meses_previstos[0]}-{meses_previstos[2]}_{ano_previsao}_ilheus.png'
chart_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fname=chart_path, dpi=CHART_DPI, bbox_inches='tight')

## Plotar Gráfico Itabuna

In [ ]:
plt.figure(figsize=(18, 10))

for produto, mark, color_prev, color_real, margem, marker_size in zip(produtos, marks, colors_prev, colors_real, margens, markers_size):
    plt.plot([8,9],[valores_reais_itb[produto][-1],previsoes_itb[produto][0]],":",color=color_prev,lw=line_size, markersize=marker_size)
    plt.plot([x for x in range(9)],valores_reais_itb[produto], marker=mark, color=color_real,lw=line_size, markersize=marker_size)
    plt.plot([9,10,11],previsoes_itb[produto],":" ,marker=mark, label=produto.title().replace('c','ç'),color=color_prev,lw=line_size,
             markersize=marker_size)

#----------------------------------------------------
plt.xticks([x for x in range(0,12)],xlabel,size=15)
plt.ylabel("Custo (R$)",size=20)
plt.yticks([y for y in range(0,limite_y,10)],[format(y,".2f").replace(".",",") for y in range(0,limite_y,10)],size=15)
plt.grid(which='major', axis='y', alpha=CHART_GRID_ALPHA)
#----------
plt.title("Previsão do valor dos produtos em Itabuna", size=15)
plt.legend(ncol=5)
chart_path = OUTPUT_DIR / 'figure' / 'produtos_itabuna' / f'previsao_produtos_{meses_previstos[0]}-{meses_previstos[2]}_{ano_previsao}_itabuna.png'
chart_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fname=chart_path, dpi=CHART_DPI, bbox_inches='tight')